In [0]:
%python
import requests
import json
import datetime

# 1. 确保在正确的 catalog 和 schema 下，并创建 Volume 文件夹
catalog_name = "youtube_dev"
spark.sql(f"USE CATALOG {catalog_name}")
spark.sql("USE SCHEMA bronze")
spark.sql("CREATE VOLUME IF NOT EXISTS earthquake_data")

# 2. 直接拉取 API 数据
url = "https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/all_day.geojson"
response = requests.get(url)

if response.status_code == 200:
    data = response.json()
    
    # 3. 按日期命名文件，并直接以 JSON 格式原封不动地塞进 Volume 存储中
    current_date = datetime.datetime.now().strftime("%Y-%m-%d")
    target_path = f"/Volumes/{catalog_name}/bronze/earthquake_data/earthquake_data_{current_date}.json"
    
    dbutils.fs.put(
        target_path,
        json.dumps(data),
        overwrite=True
    )
    print(f"成功！原始 JSON 数据已落地至 Volume 存储：{target_path}")
    print("现在你可以完美衔接视频下一步的 Auto Loader 或 DLT 操作了！")
else:
    print("API请求失败")